In [ ]:
# Setup: imports and configuration
import sys
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display, Markdown

# Add src to path for library imports
sys.path.insert(0, str(Path("..").resolve() / "src"))

import fvs_tools as fvs
from fvs_tools.monte_carlo import load_mc_results

# Paths
RESULTS_DB = Path("../outputs/large_mc/mc_results.db")
IMAGES_DIR = Path("../docs/ReportImages")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(f"✓ Imports loaded")
print(f"✓ Results database: {RESULTS_DB}")
print(f"✓ Images directory: {IMAGES_DIR}")

# Overview and guide to the Monte Carlo FVS codebase

This report gives an overview of Monte Carlo methods, their applications in the FVS context, and an guide to the implementation covered in the repository: https://github.com/RoopsyDaisy/fors591

---

## 1. Introduction to Monte Carlo Methods

### 1.1 What is Monte Carlo Simulation?

Monte Carlo simulation is a computational technique that uses repeated random sampling to obtain numerical results. Named after the famous Monaco casino, the method harnesses randomness to solve problems that might be deterministic in principle but are too complex for analytical solutions.

The core concept is elegantly simple:

1. **Define the problem** with uncertain inputs characterized by probability distributions
2. **Randomly sample** input values from those distributions
3. **Run the model** for each sampled combination
4. **Aggregate results** to characterize the output distribution

Rather than asking "What will happen?", Monte Carlo asks "What are the possible outcomes, and how likely is each?"

The method's power comes from the **Law of Large Numbers**: as sample size increases, the sample mean converges to the true expected value. With modern computing power, we can run many simulations in minutes, building robust statistical pictures of complex systems.

Using the tools outlined here we can run 100 simulations of 268 stands in Lubrecht in under an hour using 20 cores. That's 26,800 FVS runs over 100 years (10-year intervals) with options enabled similar to those used in Assignment 5.

### 1.2 Mathematical Foundation

For a model $f$ with uncertain inputs $\mathbf{X} = (X_1, X_2, ..., X_k)$, Monte Carlo estimation approximates:

$$E[f(\mathbf{X})] \approx \frac{1}{N} \sum_{i=1}^{N} f(\mathbf{x}_i)$$

where $\mathbf{x}_i$ are samples drawn from the joint distribution of inputs.

The standard error of this estimate decreases as $1/\sqrt{N}$, meaning:
- 100 samples → ~10% precision
- 10,000 samples → ~1% precision
- Diminishing returns beyond a certain point

**Note:** This approach assumes you know the input parameter uncertainty, and assumes FVS is accurate given a set of parameters—neither of which is likely to be perfectly true in practice.

---

## 2. Why Monte Carlo for Forest Modeling

### 2.1 Sources of Uncertainty in FVS

The Forest Vegetation Simulator is a powerful empirical growth model, but it relies on numerous parameters that are inherently uncertain:

**Biological Uncertainty:**
- Background mortality rates vary with climate, pests, and disease
- Growth calibration factors depend on site-specific conditions
- Regeneration success rates fluctuate year to year

**Management Uncertainty:**
- Exact thinning intensities depend on operator decisions
- Harvest thresholds may shift with market conditions
- Treatment timing is subject to operational constraints

**Structural Uncertainty:**
- Model equations are empirical approximations
- Parameter estimates have confidence intervals
- Extrapolation beyond training data adds error

A single deterministic FVS run implicitly assumes perfect knowledge of all these factors—an assumption we know to be false.

### 2.2 The Case for Probabilistic Projections

Consider a forest manager asking: "Will this stand maintain 40% canopy cover over the next 50 years under my proposed management plan?"

A deterministic projection might answer "Yes, minimum canopy is 52%." But this hides the uncertainty. What if mortality is 20% higher than assumed? What if the thinning crew removes more volume than specified?

Monte Carlo simulation can provide:
- "Under baseline assumptions, minimum canopy is 52%"
- "90% of simulations stay above 45% canopy"
- "5% of simulations drop below 40% canopy"
- "Risk of threshold violation is most sensitive to mortality assumptions"

### 2.3 Applications in Forest Management

| Application | Question Addressed |
|-------------|-------------------|
| **Uncertainty Quantification** | How confident are we in carbon sequestration projections? |
| **Risk Assessment** | What's the probability of violating habitat constraints? |
| **Sensitivity Analysis** | Which parameters most influence harvest volume? |
| **Robust Optimization** | Which strategy performs well across many scenarios? |
| **Scenario Planning** | How do outcomes differ under alternative futures? |

---

## 3. Framework Architecture

### 3.1 System Overview

The Monte Carlo framework wraps the FVS simulation engine with a parameter sampling and batch execution layer:

In [ ]:
# Display the system architecture diagram
architecture_img = IMAGES_DIR / "Architecture.png"
if architecture_img.exists():
    display(Image(filename=str(architecture_img), width=700))
    print("Figure 1: Monte Carlo FVS System Architecture")
else:
    print(f"⚠️ Architecture image not found: {architecture_img}")

### 3.2 Component Overview

```
fvs_tools/
├── __init__.py           # Public API exports
├── config.py             # FVSSimulationConfig dataclass
├── runner.py             # Single FVS execution
├── batch.py              # Multi-stand batch processing
├── monte_carlo.py        # MC sampling and orchestration
├── results.py            # Output parsing and aggregation
└── database.py           # SQLite result storage
```

**Key Design Principles:**
- **Separation of concerns:** Sampling logic is independent of FVS execution
- **Reproducibility:** Seeded random number generators ensure identical results
- **Fault tolerance:** Failed runs are logged without stopping the batch
- **Parallelism:** Multi-process execution saturates available CPU cores

### 3.3 Data Flow

1. **Configuration** → User specifies parameter distributions and sample count
2. **Sampling** → Framework generates N parameter combinations
3. **Execution** → Each combination runs through the full FVS pipeline
4. **Aggregation** → Per-run metrics are computed from raw outputs
5. **Storage** → All results persist to structured database
6. **Analysis** → User loads results for visualization and interpretation

---

## 4. Parameter Specification

### 4.1 The ParameterSpec System

Parameters are specified using a flexible `ParameterSpec` system that defines both the parameter name and its sampling distribution:

In [ ]:
# Example: Defining a parameter specification
from fvs_tools.monte_carlo import UniformParameterSpec

# Define a parameter with uniform distribution
mortality_spec = UniformParameterSpec(
    name="mortality_multiplier",
    min_value=0.8,
    max_value=1.2
)

print(f"Parameter: {mortality_spec.name}")
print(f"Distribution: Uniform({mortality_spec.min_value}, {mortality_spec.max_value})")

# Demonstrate what samples from this distribution look like
rng = np.random.default_rng(42)
samples = rng.uniform(mortality_spec.min_value, mortality_spec.max_value, size=5)
print(f"Example samples: {[f'{s:.3f}' for s in samples]}")

The `name` attribute must match a field in `FVSSimulationConfig`:

```python
@dataclass
class FVSSimulationConfig:
    name: str
    num_years: int = 100
    
    # Thinning parameters
    thin_q_factor: float = 2.0
    thin_residual_ba: float = 65.0
    thin_trigger_ba: float = 100.0
    
    # Mortality
    mortality_multiplier: float = 1.0
    
    # Harvest economics
    min_harvest_volume: float = 4500.0
    
    # ... additional fields
```

### 4.2 Currently Supported Parameters

| Parameter | Type | Default | Typical Range | Effect |
|-----------|------|---------|---------------|--------|
| `thin_q_factor` | float | 2.0 | 1.5 - 2.5 | Controls diameter distribution after thinning (BDq method) |
| `thin_residual_ba` | float | 65.0 | 50 - 80 | Target basal area after thinning (ft²/ac) |
| `thin_trigger_ba` | float | 100.0 | 80 - 130 | Basal area threshold to initiate harvest |
| `mortality_multiplier` | float | 1.0 | 0.5 - 1.5 | Scalar applied to background mortality rates |
| `min_harvest_volume` | float | 4500.0 | 3000 - 6000 | Minimum board feet to justify harvest entry |

### 4.3 Example: 5-Parameter Batch Configuration

In [ ]:
# Complete Monte Carlo configuration example
from fvs_tools.monte_carlo import MonteCarloConfig, UniformParameterSpec
from fvs_tools.config import FVSSimulationConfig

# Define baseline configuration
base_config = FVSSimulationConfig(
    name="sensitivity_study",
    num_years=100,
    thin_q_factor=2.0,
    thin_residual_ba=65.0,
    thin_trigger_ba=100.0,
    mortality_multiplier=1.0,
    min_harvest_volume=4500.0,
)

# Define parameter distributions
parameter_specs = [
    UniformParameterSpec("thin_q_factor", 1.5, 2.5),
    UniformParameterSpec("thin_residual_ba", 50.0, 75.0),
    UniformParameterSpec("thin_trigger_ba", 95.0, 125.0),
    UniformParameterSpec("mortality_multiplier", 0.8, 1.2),
    UniformParameterSpec("min_harvest_volume", 3500.0, 5500.0),
]

# Create Monte Carlo configuration
mc_config = MonteCarloConfig(
    batch_seed=42,           # For reproducibility
    n_samples=100,           # Number of simulations
    n_workers=20,            # Parallel workers
    parameter_specs=parameter_specs,
    base_config=base_config,
)

print("Monte Carlo Configuration:")
print(f"  Batch seed: {mc_config.batch_seed}")
print(f"  N samples: {mc_config.n_samples}")
print(f"  N workers: {mc_config.n_workers}")
print(f"  Parameters varied: {len(parameter_specs)}")
for spec in parameter_specs:
    print(f"    - {spec.name}: [{spec.min_value}, {spec.max_value}]")

### 4.4 Sampling Strategy

The current implementation uses **simple random sampling** with independent draws for each parameter. Each run receives a unique `run_seed` derived from the `batch_seed`:

```python
# Pseudocode for parameter sampling
def sample_parameters(mc_config, run_index):
    run_seed = mc_config.batch_seed + run_index
    rng = numpy.random.default_rng(run_seed)
    
    params = {}
    for spec in mc_config.parameter_specs:
        params[spec.name] = spec.sample(rng)
    
    return params, run_seed
```

**Future Enhancement:** Latin Hypercube Sampling (LHS) would provide better coverage of the parameter space with fewer samples, improving efficiency for high-dimensional problems.

---

## 5. Batch Execution System

### 5.1 Parallel Execution Architecture

The batch system uses Python's `multiprocessing` module to execute FVS runs in parallel:

```python
from multiprocessing import Pool

def run_monte_carlo_batch(mc_config, stands, trees, output_dir):
    """Execute Monte Carlo batch with parallel workers."""
    
    # Generate all parameter samples
    run_configs = []
    for i in range(mc_config.n_samples):
        params, run_seed = sample_parameters(mc_config, i)
        config = replace(mc_config.base_config, **params)
        run_configs.append((i, config, run_seed))
    
    # Execute in parallel
    with Pool(processes=mc_config.n_workers) as pool:
        results = pool.starmap(execute_single_run, run_configs)
    
    # Aggregate and store results
    store_results(results, output_dir)
```

### 5.2 Performance Characteristics

Batch execution performance scales with available CPU cores:

| Configuration | Time (100 runs, 11 stands, 100 years) |
|---------------|---------------------------------------|
| Sequential (1 worker) | ~50 minutes |
| Parallel (10 workers) | ~6 minutes |
| Parallel (20 workers) | ~3.5 minutes |

The limiting factor is typically I/O (database writes) rather than CPU. Using SSDs significantly improves performance.

### 5.3 Fault Tolerance

The system is designed to complete the batch even if individual runs fail:

```python
# From runner.py - success detection
fvs_success = (
    "STOP 20" in result.stderr or    # Normal completion
    "STOP 10" in result.stderr or    # Completed with warnings
    result.returncode == 0
)

if not fvs_success:
    log_error(run_id, result.stderr)
    # Continue with remaining runs
```

Failed runs are logged to `MC_BatchErrors` table with full error messages for debugging.

---

## 6. Results Database Schema

### 6.1 Database Structure

All results are stored in a SQLite database (`mc_results.db`) with five tables:

In [ ]:
# Display the database schema diagram
schema_img = IMAGES_DIR / "Outputs.png"
if schema_img.exists():
    display(Image(filename=str(schema_img), width=700))
    print("Figure 2: Monte Carlo Results Database Schema")
else:
    print(f"⚠️ Schema image not found: {schema_img}")

### 6.2 Table Descriptions

**MC_BatchMeta** - Batch-level configuration and status
```sql
CREATE TABLE MC_BatchMeta (
    batch_id TEXT PRIMARY KEY,
    n_samples INTEGER,
    n_workers INTEGER,
    status TEXT,  -- 'running', 'completed', 'failed'
    created_at TIMESTAMP,
    completed_at TIMESTAMP,
    config_json TEXT  -- Full configuration for reproducibility
);
```

**MC_RunRegistry** - Sampled parameter values for each run
```sql
CREATE TABLE MC_RunRegistry (
    batch_id TEXT,
    run_id TEXT,
    run_seed INTEGER,
    thin_q_factor REAL,
    thin_residual_ba REAL,
    thin_trigger_ba REAL,
    mortality_multiplier REAL,
    min_harvest_volume REAL,
    PRIMARY KEY (batch_id, run_id)
);
```

**MC_RunSummary** - Aggregated outcome metrics per run
```sql
CREATE TABLE MC_RunSummary (
    batch_id TEXT,
    run_id TEXT,
    final_total_carbon REAL,
    avg_carbon_stock REAL,
    min_canopy_cover REAL,
    final_canopy_cover REAL,
    cumulative_harvest_bdft REAL,
    PRIMARY KEY (batch_id, run_id)
);
```

**MC_TimeSeries** - Year-by-year trajectories
```sql
CREATE TABLE MC_TimeSeries (
    batch_id TEXT,
    run_id TEXT,
    year INTEGER,
    total_carbon REAL,
    canopy_cover_pct REAL,
    ba REAL,
    cumulative_harvest REAL,
    PRIMARY KEY (batch_id, run_id, year)
);
```

**MC_BatchErrors** - Failed run diagnostics
```sql
CREATE TABLE MC_BatchErrors (
    batch_id TEXT,
    run_id TEXT,
    stand_id TEXT,
    error_msg TEXT,
    timestamp TIMESTAMP
);
```

### 6.3 Loading Results

In [ ]:
# Load results from the large_mc batch
if RESULTS_DB.exists():
    results = load_mc_results(RESULTS_DB)
    
    # Access individual components
    batch_meta = results['batch_meta']      # dict
    registry = results['registry']          # DataFrame
    summary = results['summary']            # DataFrame
    timeseries = results['timeseries']      # DataFrame
    errors = results['errors']              # DataFrame
    
    print(f"✓ Loaded {len(registry)} runs")
    print(f"  Time series: {len(timeseries)} year-observations")
    print(f"  Failed runs: {len(errors)}")
else:
    print(f"⚠️ Results database not found: {RESULTS_DB}")
    print("   Run scripts/run_large_mc.py first to generate results.")

---

## 7. Output Metrics and Aggregation

### 7.1 Understanding Flow vs Pool Variables

FVS outputs contain two fundamentally different variable types that require different aggregation approaches:

**Pool Variables (State at a Point in Time)**
- Represent the system state at the end of each period
- Examples: Basal Area, Carbon Stocks, Canopy Cover, Trees Per Acre
- Aggregation: Use values at specific time points (initial, final, min, max, mean)

**Flow Variables (Activity During a Period)**
- Represent quantities that accumulate during each period
- Examples: Harvest Volume (RBdFt), Mortality Volume, Growth
- Aggregation: Sum across periods for cumulative totals

**Critical Distinction Example:**

```python
# WRONG - treating harvest like a pool variable
final_harvest = timeseries[timeseries['year'] == 2123]['harvest_bdft'].values[0]
# This gives only the LAST period's harvest, not the total!

# CORRECT - summing flow across all periods
cumulative_harvest = timeseries['harvest_bdft'].sum()
# This gives total harvest over the entire projection
```

### 7.2 Summary Metrics

Each run produces a set of summary metrics computed from the time series:

| Metric | Type | Computation |
|--------|------|-------------|
| `final_total_carbon` | Pool | Value at final year |
| `avg_carbon_stock` | Pool | Mean across all years |
| `min_canopy_cover` | Pool | Minimum across all years |
| `final_canopy_cover` | Pool | Value at final year |
| `cumulative_harvest_bdft` | Flow | Sum across all periods |

In [ ]:
# Show summary statistics for outcomes
if RESULTS_DB.exists():
    outcome_cols = [
        'final_total_carbon', 'avg_carbon_stock',
        'min_canopy_cover', 'final_canopy_cover',
        'cumulative_harvest_bdft'
    ]
    
    print("Outcome Metrics Summary (100 Monte Carlo Runs):")
    print("=" * 70)
    display(summary[outcome_cols].describe())

---

## 8. Visualization and Analysis

### 8.1 Outcome Distribution Histograms

Histograms show the distribution of outcomes across all simulations:

In [ ]:
# Visualize outcome distributions
if RESULTS_DB.exists():
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    outcome_cols = [
        'final_total_carbon', 'avg_carbon_stock',
        'min_canopy_cover', 'final_canopy_cover',
        'cumulative_harvest_bdft'
    ]

    for i, col in enumerate(outcome_cols):
        ax = axes[i]
        values = summary[col].dropna()
        
        if len(values) > 0:
            ax.hist(values, bins=30, edgecolor='black', alpha=0.7)
            ax.axvline(values.mean(), color='red', linestyle='--', 
                       label=f'Mean: {values.mean():.1f}')
            ax.set_xlabel(col.replace('_', ' ').title())
            ax.set_ylabel('Count')
            ax.set_title(col)
            ax.legend()
            ax.grid(alpha=0.3)

    axes[-1].axis('off')
    plt.tight_layout()
    
    # Save figure
    fig.savefig(IMAGES_DIR / 'outcome_distributions.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved to {IMAGES_DIR / 'outcome_distributions.png'}")
    plt.show()

Interestingly we see the canopy cover and cumulative harvest being bimodal, this seems very perculular, I'm not sure why we would see that with those outputs but not the others! While this framework would allow us to dive into this, I didn't have time to follow up on this one.

### 8.2 Parameter-Outcome Relationships

Joining the registry (inputs) with summary (outputs) enables sensitivity analysis:

In [ ]:
# Parameter vs outcome scatter plot
if RESULTS_DB.exists():
    merged = registry.merge(summary, on=['batch_id', 'run_id'], how='inner')
    
    if 'mortality_multiplier' in merged.columns and len(merged) > 1:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        scatter = ax.scatter(
            merged['mortality_multiplier'], 
            merged['final_total_carbon'],
            c=merged['thin_q_factor'],
            cmap='viridis',
            alpha=0.7,
            edgecolors='black',
            s=60
        )
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('thin_q_factor')
        
        # Add trend line
        z = np.polyfit(merged['mortality_multiplier'], merged['final_total_carbon'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(merged['mortality_multiplier'].min(), 
                             merged['mortality_multiplier'].max(), 100)
        ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
        
        ax.set_xlabel('Mortality Multiplier')
        ax.set_ylabel('Final Total Carbon (tons/ac)')
        ax.set_title('Parameter → Outcome Relationship')
        ax.legend()
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        fig.savefig(IMAGES_DIR / 'parameter_sensitivity.png', dpi=150, bbox_inches='tight')
        print(f"✓ Saved to {IMAGES_DIR / 'parameter_sensitivity.png'}")
        plt.show()

### 8.3 Spaghetti Plots

Spaghetti plots overlay all simulation trajectories to visualize the full range of possible outcomes:

In [ ]:
# Spaghetti plots: all runs overlaid
if RESULTS_DB.exists() and len(timeseries) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    run_ids = timeseries['run_id'].unique()
    n_runs = len(run_ids)
    display_runs = min(n_runs, 100)
    colors = plt.cm.viridis(np.linspace(0, 1, display_runs))
    
    print(f"Plotting {display_runs} runs (total: {n_runs})")
    
    # Total carbon
    ax = axes[0, 0]
    for i, run_id in enumerate(run_ids[:display_runs]):
        run_data = timeseries[timeseries['run_id'] == run_id].sort_values('year')
        ax.plot(run_data['year'], run_data['total_carbon'], 
                color=colors[i], alpha=0.5, linewidth=1)
    ax.set_xlabel('Year')
    ax.set_ylabel('Total Carbon (tons/ac)')
    ax.set_title(f'Total Carbon - All Runs (n={display_runs})')
    ax.grid(alpha=0.3)
    
    # Canopy cover
    ax = axes[0, 1]
    for i, run_id in enumerate(run_ids[:display_runs]):
        run_data = timeseries[timeseries['run_id'] == run_id].sort_values('year')
        ax.plot(run_data['year'], run_data['canopy_cover_pct'], 
                color=colors[i], alpha=0.5, linewidth=1)
    ax.axhline(y=40, color='red', linestyle='--', alpha=0.7, linewidth=2, label='40% threshold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Canopy Cover (%)')
    ax.set_title(f'Canopy Cover - All Runs (n={display_runs})')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Basal area
    ax = axes[1, 0]
    for i, run_id in enumerate(run_ids[:display_runs]):
        run_data = timeseries[timeseries['run_id'] == run_id].sort_values('year')
        ax.plot(run_data['year'], run_data['ba'], 
                color=colors[i], alpha=0.5, linewidth=1)
    ax.set_xlabel('Year')
    ax.set_ylabel('Basal Area (ft²/ac)')
    ax.set_title(f'Basal Area - All Runs (n={display_runs})')
    ax.grid(alpha=0.3)
    
    # Cumulative harvest
    ax = axes[1, 1]
    for i, run_id in enumerate(run_ids[:display_runs]):
        run_data = timeseries[timeseries['run_id'] == run_id].sort_values('year')
        ax.plot(run_data['year'], run_data['cumulative_harvest'], 
                color=colors[i], alpha=0.5, linewidth=1)
    ax.set_xlabel('Year')
    ax.set_ylabel('Cumulative Harvest (bdft/ac)')
    ax.set_title(f'Cumulative Harvest - All Runs (n={display_runs})')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    fig.savefig(IMAGES_DIR / 'spaghetti_plots.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved to {IMAGES_DIR / 'spaghetti_plots.png'}")
    plt.show()

### 8.4 Percentile Band Plots

Percentile bands summarize the ensemble by showing the mean trajectory with confidence intervals:

In [ ]:
# Percentile band plots
if RESULTS_DB.exists() and len(timeseries) > 0:
    ts_agg = timeseries.groupby('year').agg({
        'total_carbon': ['mean', lambda x: x.quantile(0.05), lambda x: x.quantile(0.95)],
        'canopy_cover_pct': ['mean', lambda x: x.quantile(0.05), lambda x: x.quantile(0.95)],
        'cumulative_harvest': ['mean', lambda x: x.quantile(0.05), lambda x: x.quantile(0.95)]
    }).reset_index()

    ts_agg.columns = ['year', 
                      'total_carbon_mean', 'total_carbon_p5', 'total_carbon_p95',
                      'canopy_cover_pct_mean', 'canopy_cover_pct_p5', 'canopy_cover_pct_p95',
                      'cumulative_harvest_mean', 'cumulative_harvest_p5', 'cumulative_harvest_p95']

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Total carbon
    ax = axes[0]
    ax.plot(ts_agg['year'], ts_agg['total_carbon_mean'], 
            linewidth=2, label='Mean', color='blue')
    ax.fill_between(ts_agg['year'].astype(float), 
                    ts_agg['total_carbon_p5'].astype(float),
                    ts_agg['total_carbon_p95'].astype(float),
                    alpha=0.3, label='5th-95th percentile', color='blue')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total Carbon (tons/ac)')
    ax.set_title('Mean Carbon Trajectory')
    ax.legend()
    ax.grid(alpha=0.3)

    # Canopy cover
    ax = axes[1]
    ax.plot(ts_agg['year'], ts_agg['canopy_cover_pct_mean'], 
            linewidth=2, color='green', label='Mean')
    ax.fill_between(ts_agg['year'].astype(float), 
                    ts_agg['canopy_cover_pct_p5'].astype(float),
                    ts_agg['canopy_cover_pct_p95'].astype(float),
                    alpha=0.3, color='green', label='5th-95th percentile')
    ax.axhline(y=40, color='red', linestyle='--', alpha=0.7, label='40% threshold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Canopy Cover (%)')
    ax.set_title('Mean Canopy Cover')
    ax.legend()
    ax.grid(alpha=0.3)

    # Cumulative harvest
    ax = axes[2]
    ax.plot(ts_agg['year'], ts_agg['cumulative_harvest_mean'], 
            linewidth=2, color='brown', label='Mean')
    ax.fill_between(ts_agg['year'].astype(float), 
                    ts_agg['cumulative_harvest_p5'].astype(float),
                    ts_agg['cumulative_harvest_p95'].astype(float),
                    alpha=0.3, color='brown', label='5th-95th percentile')
    ax.set_xlabel('Year')
    ax.set_ylabel('Cumulative Harvest (bdft/ac)')
    ax.set_title('Mean Cumulative Harvest')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(IMAGES_DIR / 'percentile_bands.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved to {IMAGES_DIR / 'percentile_bands.png'}")
    plt.show()

---

## 9. Usage Guide

### 9.1 Environment Setup

The framework requires a Linux environment with the FVS binary and Python dependencies:

```bash
# Clone the repository
git clone https://github.com/RoopsyDaisy/fors591.git
cd fors591

# Install dependencies with uv
uv sync

# Verify FVS binary is accessible
which FVSie  # Should return path to compiled binary
```

Alternatively, use the provided devcontainer for a pre-configured environment.

### 9.2 Complete Workflow Example

```python
#!/usr/bin/env python
"""Complete Monte Carlo workflow example."""

from pathlib import Path
import fvs_tools as fvs
from fvs_tools.config import FVSSimulationConfig
from fvs_tools.monte_carlo import (
    MonteCarloConfig,
    UniformParameterSpec,
    run_monte_carlo_batch,
    load_mc_results,
)

# 1. Load Input Data
data_dir = Path("data")
stands = fvs.load_stands(data_dir / "FVS_Lubrecht_2023_FVS_FVS_StandInit.csv")
trees = fvs.load_trees(data_dir / "FVS_Lubrecht_2023_FVS_FVS_TreeInit.csv")

# 2. Configure Monte Carlo Batch
base_config = FVSSimulationConfig(
    name="lubrecht_mc",
    num_years=100,
    thin_q_factor=2.0,
    thin_residual_ba=65.0,
    thin_trigger_ba=100.0,
    mortality_multiplier=1.0,
    min_harvest_volume=4500.0,
)

parameter_specs = [
    UniformParameterSpec("thin_q_factor", 1.5, 2.5),
    UniformParameterSpec("thin_residual_ba", 50.0, 75.0),
    UniformParameterSpec("thin_trigger_ba", 95.0, 125.0),
    UniformParameterSpec("mortality_multiplier", 0.8, 1.2),
    UniformParameterSpec("min_harvest_volume", 3500.0, 5500.0),
]

mc_config = MonteCarloConfig(
    batch_seed=42,
    n_samples=100,
    n_workers=20,
    parameter_specs=parameter_specs,
    base_config=base_config,
)

# 3. Execute Batch
output_dir = Path("outputs/lubrecht_mc")
run_monte_carlo_batch(mc_config, stands, trees, output_dir)

# 4. Analyze Results
results = load_mc_results(output_dir / "mc_results.db")
summary = results['summary']

print(f"Final Carbon: {summary['final_total_carbon'].mean():.1f} ± "
      f"{summary['final_total_carbon'].std():.1f} tons/ac")
```

### 9.3 Running Long Batches

For large batches (500+ samples), use `tmux` for persistent execution:

```bash
# Start tmux session
tmux new-session -d -s mc_batch "uv run python scripts/run_large_mc.py 2>&1 | tee mc_batch.log"

# Check progress
tmux attach -t mc_batch

# Detach without stopping: Ctrl+B, then D

# Check log without attaching
tail -f mc_batch.log
```

---

## 10. Limitations and Considerations

### 10.1 Current Limitations

**Parameter Coverage:**
- Only 5 parameters currently supported for variation
- Treatment timing is fixed (not yet variable)
- Climate scenarios not yet integrated

**Sampling Method:**
- Simple random sampling only
- Latin Hypercube Sampling would improve efficiency
- No correlated parameter sampling

### 10.2 Interpretation Caveats

**Independence Assumption:**
- Parameters are sampled independently
- In reality, some parameters may be correlated
- Results may overestimate uncertainty if correlations exist

**Distribution Choice:**
- Uniform distributions assume equal likelihood across range
- Real parameter uncertainty may be better represented by normal or triangular distributions
- Distribution choice significantly affects results

**Model Validity:**
- Monte Carlo quantifies parameter uncertainty, not model error
- If the FVS model is biased, all Monte Carlo runs share that bias
- Results are conditional on the model being structurally correct

### 10.3 Recommended Practices

1. **Start small:** Test with 10-20 samples before running hundreds
2. **Check for failures:** Always examine the errors table
3. **Validate ranges:** Ensure parameter ranges are physically meaningful
4. **Document assumptions:** Record why specific distributions were chosen
5. **Sensitivity first:** Run smaller batches varying one parameter at a time before full factorial

In [ ]:
# Final summary
if RESULTS_DB.exists():
    print("Report Complete!")
    print("=" * 50)
    print(f"\nMonte Carlo Batch Summary:")
    print(f"  Total runs: {len(registry)}")
    print(f"  Successful: {len(summary)}")
    print(f"  Failed: {len(errors)}")
    print(f"\nKey Outcomes (mean ± std):")
    print(f"  Final Carbon:     {summary['final_total_carbon'].mean():6.1f} ± {summary['final_total_carbon'].std():5.1f} tons/ac")
    print(f"  Min Canopy Cover: {summary['min_canopy_cover'].mean():6.1f} ± {summary['min_canopy_cover'].std():5.1f} %")
    print(f"  Cum Harvest:      {summary['cumulative_harvest_bdft'].mean():6.0f} ± {summary['cumulative_harvest_bdft'].std():5.0f} bdft/ac")
    print(f"\nImages saved to: {IMAGES_DIR}")
else:
    print("⚠️ Run the Monte Carlo batch first to see results.")

## Export to HTML

Export the completed notebook to HTML for submission.

In [ ]:
import subprocess
from datetime import datetime

# Get the notebook path
notebook_path = Path.cwd() / "MonteCarloReport.ipynb"
output_dir = notebook_path.parent

# Create output filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_output = output_dir.parent / f"MonteCarloReport_{timestamp}.html"

# Export to HTML using nbconvert
result = subprocess.run(
    ["uv", "run", "jupyter", "nbconvert", "--to", "html", 
     "--output", str(html_output), str(notebook_path)],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"✓ Notebook exported successfully!")
    print(f"  Output: {html_output}")
else:
    print(f"✗ Export failed:")
    print(result.stderr)